# Binary classification in author profiling
This notebook focuses on binary author profiling —a task related to identifying user characteristics such as gender and country from text data. The dataset used corresponds to the Sentiments Competition 2024 and originates from a Kaggle source. The project applies natural language processing (NLP) and deep learning methods to perform classification based on user-generated text.<br>
The architecture is based on hierarchical neural architecture that combines:
RoBERTuito Base (a Spanish language model from pysentimiento), Cross-attention mechanisms, and Optional n-gram features to predict two attributes:
- Gender (binary classification)
- Country (multi-class classification with 5–7 categories)

In [ ]:
# Hierarchical + Cross Attention + RoBERTuito Base
import torch
import torch.nn as nn
from transformers import AutoModel

class HierarchicalRobertuitoClassifier(nn.Module):
    def __init__(self, robertuito_model="pysentimiento/robertuito-base-cased", 
                 num_genders=2, num_countries=7, ngram_dim=0):
        super().__init__()

        self.bert = AutoModel.from_pretrained(robertuito_model)
        self.hidden_size = self.bert.config.hidden_size

        self.sentence_attention = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Tanh(),
            nn.Linear(self.hidden_size, 1)
        )

        self.ngram_enabled = ngram_dim > 0
        self.ngram_dim = ngram_dim

        if self.ngram_enabled:
            # Optional: transform ngram space to same hidden dim
            self.ngram_proj = nn.Linear(ngram_dim, self.hidden_size)

            # Cross attention: RoBERTuito <-> Ngram features
            self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=4, batch_first=True)

        self.dropout = nn.Dropout(0.3)
        self.gender_head = nn.Linear(self.hidden_size, num_genders)
        self.country_head = nn.Linear(self.hidden_size, num_countries)

    def forward(self, input_ids_batch, attention_mask_batch, ngram_feats_batch=None):
        # input_ids_batch: [batch, num_sents, seq_len]
        B, S, L = input_ids_batch.shape

        input_ids_flat = input_ids_batch.view(B*S, L)
        attention_mask_flat = attention_mask_batch.view(B*S, L)

        outputs = self.bert(input_ids=input_ids_flat, attention_mask=attention_mask_flat)
        sent_embs = outputs.pooler_output.view(B, S, -1)  # [B, S, H]

        # Hierarchical Attention
        attn_weights = self.sentence_attention(sent_embs)  # [B, S, 1]
        attn_weights = torch.softmax(attn_weights, dim=1)  # [B, S, 1]
        doc_emb = torch.sum(attn_weights * sent_embs, dim=1)  # [B, H]

        # Cross Attention with n-gram features (optional)
        if self.ngram_enabled and ngram_feats_batch is not None:
            ngram_proj = self.ngram_proj(ngram_feats_batch).unsqueeze(1)  # [B, 1, H]
            doc_emb, _ = self.cross_attention(query=doc_emb.unsqueeze(1), key=ngram_proj, value=ngram_proj)
            doc_emb = doc_emb.squeeze(1)  # [B, H]

        doc_emb = self.dropout(doc_emb)
        gender_logits = self.gender_head(doc_emb)
        country_logits = self.country_head(doc_emb)
        return gender_logits, country_logits


In [ ]:
# ===== PREPROCESADO =====
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Tokenizador RoBERTuito
robertuito_tokenizer = AutoTokenizer.from_pretrained("pysentimiento/robertuito-base-cased")

def tokenizar_jerarquico(user_texts, tokenizer, max_sent=10, max_len=64):
    input_ids_dict = {}
    attention_masks_dict = {}
    for user_id, sentences in user_texts.items():
        selected = sentences[:max_sent]
        input_ids = []
        att_masks = []
        for s in selected:
            encoded = tokenizer(
                " ".join(s), truncation=True, padding='max_length', max_length=max_len, return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'])
            att_masks.append(encoded['attention_mask'])

        pad_len = max_sent - len(input_ids)
        if pad_len > 0:
            input_ids += [torch.zeros((1, max_len), dtype=torch.long)] * pad_len
            att_masks += [torch.zeros((1, max_len), dtype=torch.long)] * pad_len

        input_ids_dict[user_id] = torch.cat(input_ids, dim=0)
        attention_masks_dict[user_id] = torch.cat(att_masks, dim=0)

    return input_ids_dict, attention_masks_dict

def generar_ngramas(user_texts, ngram_range=(1, 2), max_features=5000):
    corpus = []
    user_ids = []
    for uid, sents in user_texts.items():
        flat_text = " ".join([" ".join(w) for w in sents])
        corpus.append(flat_text)
        user_ids.append(uid)

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    features = vectorizer.fit_transform(corpus)
    ngram_tensor = torch.tensor(features.toarray(), dtype=torch.float32)

    return {uid: ngram_tensor[i] for i, uid in enumerate(user_ids)}, vectorizer

# Dataset adaptado para jerarquía + ngramas
from torch.utils.data import Dataset

class HierarchicalDataset(Dataset):
    def __init__(self, input_ids_dict, att_masks_dict, genders, countries, gender2id, country2id, ngram_feats=None):
        self.ids = list(input_ids_dict.keys())
        self.inputs = input_ids_dict
        self.masks = att_masks_dict
        self.genders = genders
        self.countries = countries
        self.gender2id = gender2id
        self.country2id = country2id
        self.ngram_feats = ngram_feats

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        uid = self.ids[idx]
        input_ids = self.inputs[uid]         # [S, L]
        attention_mask = self.masks[uid]     # [S, L]
        gender = self.gender2id[self.genders[uid]]
        country = self.country2id[self.countries[uid]]

        if self.ngram_feats is not None:
            return input_ids, attention_mask, gender, country, self.ngram_feats[uid]
        else:
            return input_ids, attention_mask, gender, country


In [ ]:
# Hierarchical + Cross Attention + RoBERTuito Base
#Aqui no se si le cambio algo
import torch
import torch.nn as nn
from transformers import AutoModel

class HierarchicalRobertuitoClassifier(nn.Module):
    def __init__(self, robertuito_model="pysentimiento/robertuito-base-cased", 
                 num_genders=2, num_countries=5, ngram_dim=0):
        super().__init__()

        self.bert = AutoModel.from_pretrained(robertuito_model)
        self.hidden_size = self.bert.config.hidden_size

        self.sentence_attention = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Tanh(),
            nn.Linear(self.hidden_size, 1)
        )

        self.ngram_enabled = ngram_dim > 0
        self.ngram_dim = ngram_dim

        if self.ngram_enabled:
            self.ngram_proj = nn.Linear(ngram_dim, self.hidden_size)
            self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=4, batch_first=True)

        self.dropout = nn.Dropout(0.3)
        self.gender_head = nn.Linear(self.hidden_size, num_genders)
        self.country_head = nn.Linear(self.hidden_size, num_countries)

    def forward(self, input_ids_batch, attention_mask_batch, ngram_feats_batch=None):
        B, S, L = input_ids_batch.shape

        input_ids_flat = input_ids_batch.view(B*S, L)
        attention_mask_flat = attention_mask_batch.view(B*S, L)

        outputs = self.bert(input_ids=input_ids_flat, attention_mask=attention_mask_flat)
        sent_embs = outputs.pooler_output.view(B, S, -1)  # [B, S, H]

        attn_weights = self.sentence_attention(sent_embs)  # [B, S, 1]
        attn_weights = torch.softmax(attn_weights, dim=1)  # [B, S, 1]
        doc_emb = torch.sum(attn_weights * sent_embs, dim=1)  # [B, H]

        if self.ngram_enabled and ngram_feats_batch is not None:
            ngram_proj = self.ngram_proj(ngram_feats_batch).unsqueeze(1)  # [B, 1, H]
            doc_emb, _ = self.cross_attention(query=doc_emb.unsqueeze(1), key=ngram_proj, value=ngram_proj)
            doc_emb = doc_emb.squeeze(1)  # [B, H]

        doc_emb = self.dropout(doc_emb)
        gender_logits = self.gender_head(doc_emb)
        country_logits = self.country_head(doc_emb)
        return gender_logits, country_logits

# ===== PREPROCESADO =====
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Tokenizador RoBERTuito
robertuito_tokenizer = AutoTokenizer.from_pretrained("pysentimiento/robertuito-base-cased")

def tokenizar_jerarquico(user_texts, tokenizer, max_sent=10, max_len=64):
    input_ids_dict = {}
    attention_masks_dict = {}
    for user_id, sentences in user_texts.items():
        selected = sentences[:max_sent]
        input_ids = []
        att_masks = []
        for s in selected:
            encoded = tokenizer(
                " ".join(s), truncation=True, padding='max_length', max_length=max_len, return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'])
            att_masks.append(encoded['attention_mask'])

        pad_len = max_sent - len(input_ids)
        if pad_len > 0:
            input_ids += [torch.zeros((1, max_len), dtype=torch.long)] * pad_len
            att_masks += [torch.zeros((1, max_len), dtype=torch.long)] * pad_len

        input_ids_dict[user_id] = torch.cat(input_ids, dim=0)
        attention_masks_dict[user_id] = torch.cat(att_masks, dim=0)

    return input_ids_dict, attention_masks_dict

def generar_ngramas(user_texts, ngram_range=(1, 2), max_features=5000):
    corpus = []
    user_ids = []
    for uid, sents in user_texts.items():
        flat_text = " ".join([" ".join(w) for w in sents])
        corpus.append(flat_text)
        user_ids.append(uid)

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    features = vectorizer.fit_transform(corpus)
    ngram_tensor = torch.tensor(features.toarray(), dtype=torch.float32)

    return {uid: ngram_tensor[i] for i, uid in enumerate(user_ids)}, vectorizer

# Dataset adaptado para jerarquía + ngramas
from torch.utils.data import Dataset, DataLoader

def collate_hierarchical(batch):
    input_ids, attn_masks, genders, countries, ngrams = zip(*batch)
    return (
        torch.stack(input_ids),
        torch.stack(attn_masks),
        torch.tensor(genders),
        torch.tensor(countries),
        torch.stack(ngrams)
    )

class HierarchicalDataset(Dataset):
    def __init__(self, input_ids_dict, att_masks_dict, genders, countries, gender2id, country2id, ngram_feats=None):
        self.ids = list(input_ids_dict.keys())
        self.inputs = input_ids_dict
        self.masks = att_masks_dict
        self.genders = genders
        self.countries = countries
        self.gender2id = gender2id
        self.country2id = country2id
        self.ngram_feats = ngram_feats

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        uid = self.ids[idx]
        input_ids = self.inputs[uid]         # [S, L]
        attention_mask = self.masks[uid]     # [S, L]
        gender = self.gender2id[self.genders[uid]]
        country = self.country2id[self.countries[uid]]

        if self.ngram_feats is not None:
            return input_ids, attention_mask, gender, country, self.ngram_feats[uid]
        else:
            return input_ids, attention_mask, gender, country, torch.zeros(1)
        
        
    # ENTRENAMIENTO FINAL

def entrenar(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for input_ids, attn_masks, gender_labels, country_labels, ngram_feats in dataloader:
        input_ids = input_ids.to(device)
        attn_masks = attn_masks.to(device)
        gender_labels = gender_labels.to(device)
        country_labels = country_labels.to(device)
        ngram_feats = ngram_feats.to(device)

        optimizer.zero_grad()
        gender_logits, country_logits = model(input_ids, attn_masks, ngram_feats)
        loss = loss_fn(gender_logits, gender_labels) + loss_fn(country_logits, country_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluar_conjunto(model, dataloader, device):
    model.eval()
    correct_joint = 0
    total = 0
    with torch.no_grad():
        for input_ids, attn_masks, gender_labels, country_labels, ngram_feats in dataloader:
            input_ids = input_ids.to(device)
            attn_masks = attn_masks.to(device)
            gender_labels = gender_labels.to(device)
            country_labels = country_labels.to(device)
            ngram_feats = ngram_feats.to(device)

            gender_logits, country_logits = model(input_ids, attn_masks, ngram_feats)
            gender_preds = torch.argmax(gender_logits, dim=1)
            country_preds = torch.argmax(country_logits, dim=1)

            correct = (gender_preds == gender_labels) & (country_preds == country_labels)
            correct_joint += correct.sum().item()
            total += gender_labels.size(0)
    return correct_joint / total



In [ ]:
# Entrenamiento completo del modelo jerárquico con atención cruzada y n-gramas

# Paso 1: Preprocesamiento
train_input_ids, train_attn_masks = tokenizar_jerarquico(train_texts, robertuito_tokenizer)
val_input_ids, val_attn_masks = tokenizar_jerarquico(val_texts, robertuito_tokenizer)

train_ngrams, vectorizer = generar_ngramas(train_texts)
val_ngrams = {uid: torch.tensor(vectorizer.transform([" ".join([" ".join(s) for s in val_texts[uid]])]).toarray()[0], dtype=torch.float32) for uid in val_texts}

# Paso 2: Mapas de etiquetas
gender2id = {g: i for i, g in enumerate(set(train_genders.values()))}
country2id = {c: i for i, c in enumerate(set(train_countries.values()))}

# Paso 3: Datasets y Dataloaders
train_dataset = HierarchicalDataset(train_input_ids, train_attn_masks, train_genders, train_countries, gender2id, country2id, train_ngrams)
val_dataset = HierarchicalDataset(val_input_ids, val_attn_masks, val_genders, val_countries, gender2id, country2id, val_ngrams)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_hierarchical)
val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_hierarchical)

# Paso 4: Modelo
model = HierarchicalRobertuitoClassifier(
    robertuito_model="pysentimiento/robertuito-base-cased",
    num_genders=len(gender2id),
    num_countries=len(country2id),
    ngram_dim=vectorizer.max_features
)
model.to(device)

# Paso 5: Entrenamiento
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
loss_fn = nn.CrossEntropyLoss()

best_acc = 0.0
for epoch in range(25):
    loss = entrenar(model, train_loader, optimizer, loss_fn, device)
    acc_joint = evaluar_conjunto(model, val_loader, device)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Acc: {acc_joint:.4f}")

    if acc_joint > best_acc:
        best_acc = acc_joint
        torch.save(model.state_dict(), "mejor_modelo_robertuito_hierarchical.pt")
        print(">> Guardado mejor modelo")


In [ ]:
# Evaluación de nuevos usuarios con modelo jerárquico entrenado

def predecir_usuario(model, tokenizer, vectorizer, user_text, device, max_sentences=10):
    model.eval()
    # Tokenizar oraciones
    sentences = user_text.split(".")
    sentences = [s.strip() for s in sentences if s.strip()]
    if len(sentences) > max_sentences:
        sentences = sentences[:max_sentences]

    inputs = [tokenizer(s, truncation=True, padding='max_length', max_length=512, return_tensors='pt') for s in sentences]
    input_ids = torch.stack([x['input_ids'].squeeze(0) for x in inputs])  # [N, 512]
    attn_masks = torch.stack([x['attention_mask'].squeeze(0) for x in inputs])

    # Padding si hay menos de max_sentences
    num_missing = max_sentences - input_ids.size(0)
    if num_missing > 0:
        pad_ids = torch.zeros((num_missing, 512), dtype=torch.long)
        pad_masks = torch.zeros((num_missing, 512), dtype=torch.long)
        input_ids = torch.cat([input_ids, pad_ids], dim=0)
        attn_masks = torch.cat([attn_masks, pad_masks], dim=0)

    # TF-IDF n-gramas
    text_joined = " ".join(sentences)
    ngram_vector = torch.tensor(vectorizer.transform([text_joined]).toarray()[0], dtype=torch.float32)

    input_ids = input_ids.unsqueeze(0).to(device)
    attn_masks = attn_masks.unsqueeze(0).to(device)
    ngram_vector = ngram_vector.unsqueeze(0).to(device)

    with torch.no_grad():
        logits_gender, logits_country = model(input_ids, attn_masks, ngram_vector)
        pred_gender = torch.argmax(logits_gender, dim=1).item()
        pred_country = torch.argmax(logits_country, dim=1).item()

    return pred_gender, pred_country

# Uso:
# pred_gender_id, pred_country_id = predecir_usuario(model, robertuito_tokenizer, vectorizer, texto_usuario, device)
# print("Predicción género:", list(gender2id.keys())[list(gender2id.values()).index(pred_gender_id)])
# print("Predicción país:", list(country2id.keys())[list(country2id.values()).index(pred_country_id)])


In [ ]:
# Texto simulado de un nuevo usuario (puedes usar fragmentos reales)
nuevo_usuario_texto = """
Hoy me levanté muy temprano para ir a correr. Me encanta comenzar el día con energía.
Después del trabajo, estuve revisando noticias sobre política internacional. 
Siempre me ha interesado cómo se manejan las relaciones entre países. 
Además, estoy aprendiendo a cocinar platos típicos de mi país, es una forma de mantener viva la cultura.
"""

# Predecir género y país
pred_gender_id, pred_country_id = predecir_usuario(
    model,
    robertuito_tokenizer,
    vectorizer,
    nuevo_usuario_texto,
    device
)

# Mostrar etiquetas reales
id2gender = {v: k for k, v in gender2id.items()}
id2country = {v: k for k, v in country2id.items()}

print("✅ Predicción de género:", id2gender[pred_gender_id])
print("🌍 Predicción de país:", id2country[pred_country_id])


------------------------
Primero preentrenar con solo el vocabulario para que sepa que hay en el 

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset
import random

# Crear dataset de tweets (solo texto)
texts = [" ".join([" ".join(sent) for sent in sents]) for sents in train_texts.values()]
mlm_dataset = Dataset.from_dict({"text": texts})

# Tokenización
tokenized = mlm_dataset.map(lambda ex: robertuito_tokenizer(ex["text"], truncation=True, padding="max_length", max_length=128), batched=True)

# Preparar data collator para MLM
data_collator = DataCollatorForLanguageModeling(tokenizer=robertuito_tokenizer, mlm=True, mlm_probability=0.15)

training_args = TrainingArguments(
    output_dir="./dapt-robertuito",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_steps=100,
    logging_steps=50,
    learning_rate=5e-5,
    prediction_loss_only=True
)

# Trainer para MLM
trainer = Trainer(
    model=robertuito_model,  # el modelo base
    args=training_args,
    train_dataset=tokenized,
    tokenizer=robertuito_tokenizer,
    data_collator=data_collator
)

trainer.train()


In [ ]:
from torch.nn import TransformerEncoderLayer, TransformerEncoder
#cambiar Hierechal por interClass 
class InterSentenceClassifier(nn.Module):
    def __init__(self, encoder, hidden_dim, ngram_dim, num_genders, num_countries):
        super().__init__()
        self.encoder = encoder
        self.self_attn_layer = TransformerEncoder(
            TransformerEncoderLayer(d_model=hidden_dim, nhead=4, dim_feedforward=512),
            num_layers=1
        )
        self.cross_ff = nn.Linear(hidden_dim + ngram_dim, hidden_dim)
        self.gender_out = nn.Linear(hidden_dim, num_genders)
        self.country_out = nn.Linear(hidden_dim, num_countries)

    def forward(self, input_ids, attn_masks, ngram_feats):
        # input_ids: [B, S, T]  S=sentences, T=tokens
        B, S, T = input_ids.shape
        outputs = []
        for i in range(S):
            out = self.encoder(input_ids[:, i], attention_mask=attn_masks[:, i]).pooler_output
            outputs.append(out)
        sent_repr = torch.stack(outputs, dim=1)  # [B, S, H]
        
        # Self-attention entre oraciones
        sent_attn = self.self_attn_layer(sent_repr)  # [B, S, H]
        doc_repr = sent_attn.mean(dim=1)  # o usar attention pooling

        # Cross-interaction con n-gramas
        combined = torch.cat((doc_repr, ngram_feats), dim=1)
        fused = self.cross_ff(combined)

        return self.gender_out(fused), self.country_out(fused)


-----------------

In [ ]:
# =========================
# 1. DOMAIN ADAPTIVE PRETRAINING (DAPT)
# =========================
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Cargar RoBERTuito para MLM
model_name = "pysentimiento/robertuito-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
mlm_model = AutoModelForMaskedLM.from_pretrained(model_name)

# Preparar dataset sin etiquetas
texts = [" ".join(" ".join(sent) for sent in sents) for sents in train_texts.values()]
dataset = Dataset.from_dict({"text": texts})

# Tokenización
encoded_dataset = dataset.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128), batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./dapt_robertuito",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    logging_steps=100,
    learning_rate=5e-5,
    prediction_loss_only=True
)

trainer = Trainer(
    model=mlm_model,
    args=training_args,
    train_dataset=encoded_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Entrenar DAPT
# trainer.train()  # <- Descomentar para entrenar


# =========================
# 2. InterSentenceClassifier (con atención entre oraciones y fusión n-gramas)
# =========================
import torch
import torch.nn as nn
from transformers import AutoModel

class InterSentenceClassifier(nn.Module):
    def __init__(self, base_model_name, hidden_dim, ngram_feat_dim, num_genders, num_countries):
        super().__init__()
        self.bert = AutoModel.from_pretrained(base_model_name)
        self.self_attn = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=4, dim_feedforward=512),
            num_layers=1
        )
        self.linear_fuse = nn.Linear(hidden_dim + ngram_feat_dim, hidden_dim)
        self.gender_head = nn.Linear(hidden_dim, num_genders)
        self.country_head = nn.Linear(hidden_dim, num_countries)

    def forward(self, input_ids, attention_masks, ngram_feats):
        B, S, T = input_ids.size()
        reps = []
        for i in range(S):
            out = self.bert(input_ids[:, i], attention_mask=attention_masks[:, i])
            cls = out.pooler_output
            reps.append(cls)
        sent_matrix = torch.stack(reps, dim=1)  # [B, S, H]
        attended = self.self_attn(sent_matrix)  # [B, S, H]
        doc_repr = attended.mean(dim=1)  # [B, H]
        combined = torch.cat([doc_repr, ngram_feats], dim=1)
        fused = self.linear_fuse(combined)
        return self.gender_head(fused), self.country_head(fused)


# =========================
# 3. STACKING (meta-modelo)
# =========================
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# logits_model_b, logits_model_c deben ser arrays [N, num_classes]
# y_true debe ser el array de etiquetas verdaderas

def entrenar_meta_modelo(logits_model_b, logits_model_c, y_true):
    meta_features = np.concatenate([logits_model_b, logits_model_c], axis=1)
    meta_clf = LogisticRegression()
    meta_clf.fit(meta_features, y_true)
    return meta_clf

def predecir_meta_modelo(meta_modelo, logits_b, logits_c):
    features = np.concatenate([logits_b, logits_c], axis=1)
    return meta_modelo.predict(features)

# Ejemplo de uso:
# meta_model = entrenar_meta_modelo(gender_logits_b, gender_logits_c, y_gender)
# preds = predecir_meta_modelo(meta_model, gender_logits_b_test, gender_logits_c_test)
# acc = accuracy_score(y_gender_test, preds)
# print("Meta-accuracy:", acc)
# =========================
# 4. INFERENCIA CON STACKING
# =========================
def obtener_logits(model, dataloader, device):
    model.eval()
    gender_logits_all = []
    country_logits_all = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids, attention_mask, gender_labels, country_labels, ngram_feats = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            ngram_feats = ngram_feats.to(device)

            gender_logits, country_logits = model(input_ids, attention_mask, ngram_feats)
            gender_logits_all.append(gender_logits.cpu())
            country_logits_all.append(country_logits.cpu())

    return torch.cat(gender_logits_all).numpy(), torch.cat(country_logits_all).numpy()


def evaluar_stacking(model_b, model_c, dataloader, y_gender, y_country, device):
    logits_b_gender, logits_b_country = obtener_logits(model_b, dataloader, device)
    logits_c_gender, logits_c_country = obtener_logits(model_c, dataloader, device)

    meta_gender = entrenar_meta_modelo(logits_b_gender, logits_c_gender, y_gender)
    meta_country = entrenar_meta_modelo(logits_b_country, logits_c_country, y_country)

    preds_gender = predecir_meta_modelo(meta_gender, logits_b_gender, logits_c_gender)
    preds_country = predecir_meta_modelo(meta_country, logits_b_country, logits_c_country)

    acc_gender = accuracy_score(y_gender, preds_gender)
    acc_country = accuracy_score(y_country, preds_country)
    acc_joint = ((preds_gender == y_gender) & (preds_country == y_country)).mean()

    print(f"Stacking - Gender Accuracy: {acc_gender:.4f}")
    print(f"Stacking - Country Accuracy: {acc_country:.4f}")
    print(f"Stacking - Joint Accuracy: {acc_joint:.4f}")

In [ ]:
# =========================
# 5. PREDICCIÓN PARA UN NUEVO USUARIO CON STACKING
# =========================
def predecir_usuario_stacking(model_b, model_c, meta_gender, meta_country, tokenizer, vectorizer, texto_usuario, device):
    # Tokenización y features
    tokens = tokenizer(texto_usuario, truncation=True, padding='max_length', max_length=512, return_tensors='pt')
    input_ids = tokens['input_ids'].unsqueeze(0).to(device)
    attention_mask = tokens['attention_mask'].unsqueeze(0).to(device)
    
    # TF-IDF
    tfidf_feats = vectorizer.transform([texto_usuario])
    tfidf_tensor = torch.tensor(tfidf_feats.toarray(), dtype=torch.float32).to(device)

    # Obtener logits de ambos modelos
    model_b.eval()
    model_c.eval()
    with torch.no_grad():
        logits_b_gender, logits_b_country = model_b(input_ids, attention_mask, tfidf_tensor)
        logits_c_gender, logits_c_country = model_c(input_ids, attention_mask, tfidf_tensor)

    # Stacking
    logits_b_gender = logits_b_gender.cpu().numpy()
    logits_c_gender = logits_c_gender.cpu().numpy()
    logits_b_country = logits_b_country.cpu().numpy()
    logits_c_country = logits_c_country.cpu().numpy()

    pred_gender = predecir_meta_modelo(meta_gender, logits_b_gender, logits_c_gender)[0]
    pred_country = predecir_meta_modelo(meta_country, logits_b_country, logits_c_country)[0]

    return pred_gender, pred_country